In [1]:
from langchain_openai import ChatOpenAI
from os import environ

In [2]:
llm = ChatOpenAI(
    model="openai.gpt-4o",
    temperature=0.2,
)

In [3]:
from langchain_core.messages import HumanMessage

llm.invoke([HumanMessage(content="Hi! I'm Bob")])

AIMessage(content='Hi Bob! 👋 Nice to meet you. How can I help you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 11, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_f3927aa00d', 'finish_reason': 'stop', 'logprobs': None}, id='run-a311c77f-78be-48ef-8987-42e088b3ff3f-0', usage_metadata={'input_tokens': 11, 'output_tokens': 18, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

<h2>Load Source Text</h2>

In [4]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("/workspace/data/knowledge_base/fruits_and_veggies.txt")
documents = loader.load()

In [8]:
documents[0].metadata

{'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}

In [9]:
print(documents[0].page_content)

The Amanita phalloides has a large and imposing epigeous (above ground) fruiting body (basidiocrap). 
A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all white. 
AA. Phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.
Gala apples are a popular variety known for their sweet flavor and crisp texture. 
They have a distinctive reddish-orange skin with yellow striping, making them visually appealing in fruit displays. 
Originally developed in New Zealand in the 1930s, they have since become a favorite in many countries and are widely cultivated for consumption. 
Their versatility makes them perfect for both eating fresh and using in various culinary dishes.
Radishes are small, root vegetables with a sharp, peppery flavor that can range from mild to spicy. 
They are usually round or cylindrical in shape and can come in various colors, including red, white, purple, and black. 
Rich in vitamins and minerals, radishes are often c

<h2>Split the document</h2>

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
chunk_size = 100
chunk_overlap = 0

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
)

In [13]:
chunks = text_splitter.split_documents(documents)

In [14]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

The Amanita phalloides has a large and imposing epigeous (above ground) fruiting body (basidiocrap).
-----
A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all white.
-----
AA. Phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.
-----
Gala apples are a popular variety known for their sweet flavor and crisp texture.
-----
They have a distinctive reddish-orange skin with yellow striping, making them visually appealing in
-----
fruit displays.
-----
Originally developed in New Zealand in the 1930s, they have since become a favorite in many
-----
countries and are widely cultivated for consumption.
-----
Their versatility makes them perfect for both eating fresh and using in various culinary dishes.
-----
Radishes are small, root vegetables with a sharp, peppery flavor that can range from mild to spicy.
-----
They are usually round or cylindrical in shape and can come in various colors, including red,
-----
white, purple, and

## Index chunks into a vector db (ChromaDB)

In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [16]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(model="openai.text-embedding-3-large"))
#vectorstore = Chroma.from_documents(documents=chunks, embedding=AzureOpenAIEmbeddings(model="text-embedding-ada-002"))

## Test Similarity Search

In [17]:
vectorstore.similarity_search("Amanita phalloides")

[Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='AA. Phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.'),
 Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='The Amanita phalloides has a large and imposing epigeous (above ground) fruiting body (basidiocrap).'),
 Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all white.'),
 Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='countries and are widely cultivated for consumption.')]

In [18]:
# Note that providers implement different scores; Chroma here
# returns a distance metric that should vary inversely with similarity.
vectorstore.similarity_search_with_score("Amanita phalloides")

[(Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='AA. Phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.'),
  0.6481142640113831),
 (Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='The Amanita phalloides has a large and imposing epigeous (above ground) fruiting body (basidiocrap).'),
  0.6505115628242493),
 (Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all white.'),
  0.667125940322876),
 (Document(metadata={'source': '/workspace/data/knowledge_base/fruits_and_veggies.txt'}, page_content='countries and are widely cultivated for consumption.'),
  1.571354627609253)]

## Prepare prompt (Augmentation Step)

In [16]:
from langchain_core.prompts import PromptTemplate

template = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    
    Question: {question} 
    
    Context: {context} 
    
    Answer:
"""
prompt = PromptTemplate.from_template(template)

## Setup retrieval

In [17]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [ ]:
	
retrieved_docs = retriever.invoke("Amanita phalloides")

len(retrieved_docs)

In [ ]:
retrieved_docs[0].page_content

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
format_docs(retriever.invoke("Amanita phalloides"))

## Build RAG chain

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("tell me about Amanita phalloides")

## Context Aware Text Splitter

In [ ]:
documents

In [ ]:
def llm_similarity(text1, text2):
    template = """
    Analyze the contextual relationship between the following two texts:
    
    Text 1: {text1}
    Text 2: {text2}
    
    Evaluate whether Text 2 completes or extends the context of Text 1, or if they are separate and unrelated. Assign a float score from 0 to 1, where:
    
    0 = The texts are entirely unrelated and should be split
    1 = The texts are strongly connected and belong to the same context
    
    Consider factors such as:
    
    Thematic continuity
    Logical flow
    Shared subject matter
    Narrative or argumentative progression
    Linguistic cohesion
    Provide only a single float value between 0 and 1 as your response, with up to two decimal places. For example: 0.75
    
    Ensure your answer contains nothing but the float value. Double-check your response before submitting"""
    
    prompt = PromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()

    similarity = chain.invoke({"text1": text1, "text2": text2})
    return float(similarity.replace('.\n\n', ''))

In [ ]:
llm_similarity("""The Amanita phalloides has a large and imposing epigeous (above ground) fruiting body (basidiocrap).""",
               """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all white.""")

In [ ]:
def merge_metadata(metadatas):
    merged_metadata = {}
    for metadata in metadatas:
        for key, value in metadata.items():
            if key in merged_metadata:
                merged_metadata[key] += " " + value  
            else:
                merged_metadata[key] = value
    return merged_metadata

In [ ]:
from langchain_core.documents import Document

def context_text_splitter_with_llm(documents, step_size, chunk_size, max_chunk_size):
    # Split the text
    # Ensure you have RecursiveCharacterTextSplitter defined and available
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=step_size, chunk_overlap=0)

    docs = text_splitter.split_documents(documents)

    step_chunks = [doc.page_content for doc in docs]
    step_metadata = [doc.metadata for doc in docs]

    merged_chunks = []
    merged_metadata_chunks = []

    while len(step_chunks) > 0:
        if len(''.join(step_chunks)) < chunk_size:
            break
        
        current_chunk = ''.join(step_chunks[:chunk_size//step_size])
        current_metadata = merge_metadata(step_metadata[:chunk_size//step_size])
        
        step_chunks = step_chunks[chunk_size//step_size:]
        step_metadata = step_metadata[chunk_size//step_size:]

        chunk_appended = False

        while len(step_chunks) > 0:
            next_step_chunk = step_chunks.pop(0)
            next_step_chunk_metadata = step_metadata.pop(0)

            similarity_score = llm_similarity(current_chunk, next_step_chunk)

            if similarity_score > 0.49 and len(current_chunk) + len(next_step_chunk) <= max_chunk_size:
                current_chunk += " " + next_step_chunk
                current_metadata = merge_metadata([current_metadata, next_step_chunk_metadata])
            else:
                merged_chunks.append(" " + current_chunk)
                merged_metadata_chunks.append(current_metadata)
                
                chunk_appended = True
                
                step_chunks.insert(0, next_step_chunk)
                step_metadata.insert(0, next_step_chunk_metadata)
                
                break

        if not chunk_appended:
            merged_chunks.append(" " + current_chunk)
            merged_metadata_chunks.append(current_metadata)

    if len(step_chunks) > 0:
        merged_chunks.append(' '.join(step_chunks))
        merged_metadata_chunks.append(merge_metadata(step_metadata))

    merged_docs = []
    for chunk, metadata in zip(merged_chunks, merged_metadata_chunks):
        merged_docs.append(Document(page_content=chunk, metadata=metadata))

    return merged_docs

In [ ]:
chunks = context_text_splitter_with_llm(documents, 100, 200, 1200)

In [ ]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

## Index into Chroma DB

In [ ]:
import chromadb

# Initialize ChromaDB client
client = chromadb.Client()

# Access a specific collection
collection_name = "langchain"
client.delete_collection(collection_name)

In [ ]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=AzureOpenAIEmbeddings(model="text-embedding-3-large"))

## Setup Retrieval

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("tell me about Amanita phalloides")

## Bonus Vector Similarity

In [ ]:
embeddings = AzureOpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
v1 = embeddings.embed_documents(texts=["Apple"])[0]
v2 = embeddings.embed_documents(texts=["Orange"])[0]

In [ ]:
from scipy.spatial.distance import cosine
def cosine_similarity(vec1, vec2):
    """Compute the cosine similarity between two vectors using SciPy."""
    return 1 - cosine(vec1, vec2)  # cosine function from SciPy computes the distance, not similarity

In [ ]:
cosine_similarity(v1, v2)

In [ ]:
from langchain_core.documents import Document
def context_text_splitter(documents, step_size, chunk_size, max_chunk_size):
    # Split the text
    # Ensure you have RecursiveCharacterTextSplitter defined and available
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=step_size, chunk_overlap=0)

    docs = text_splitter.split_documents(documents)

    step_chunks = [doc.page_content for doc in docs]
    step_metadata = [doc.metadata for doc in docs]

    merged_chunks = []
    merged_metadata_chunks = []

    while len(step_chunks) > 0:
        if len(''.join(step_chunks)) < chunk_size:
            break
        
        current_chunk = ''.join(step_chunks[:chunk_size//step_size])
        current_metadata = merge_metadata(step_metadata[:chunk_size//step_size])
        
        step_chunks = step_chunks[chunk_size//step_size:]
        step_metadata = step_metadata[chunk_size//step_size:]

        chunk_appended = False

        while len(step_chunks) > 0:
            next_step_chunk = step_chunks.pop(0)
            next_step_chunk_metadata = step_metadata.pop(0)

            similarity_score = cosine_similarity(embeddings.embed_query(current_chunk), embeddings.embed_query(next_step_chunk))

            if similarity_score > 0.79 and len(current_chunk) + len(next_step_chunk) <= max_chunk_size:
                current_chunk += " " + next_step_chunk
                current_metadata = merge_metadata([current_metadata, next_step_chunk_metadata])
            else:
                merged_chunks.append(" " + current_chunk)
                merged_metadata_chunks.append(current_metadata)
                
                chunk_appended = True
                
                step_chunks.insert(0, next_step_chunk)
                step_metadata.insert(0, next_step_chunk_metadata)
                
                break

        if not chunk_appended:
            merged_chunks.append(" " + current_chunk)
            merged_metadata_chunks.append(current_metadata)

    if len(step_chunks) > 0:
        merged_chunks.append(' '.join(step_chunks))
        merged_metadata_chunks.append(merge_metadata(step_metadata))

    merged_docs = []
    for chunk, metadata in zip(merged_chunks, merged_metadata_chunks):
        merged_docs.append(Document(page_content=chunk, metadata=metadata))

    return merged_docs

In [ ]:
chunks = context_text_splitter(documents, 100, 200, 1200)

In [ ]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

In [ ]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=AzureOpenAIEmbeddings(model="text-embedding-3-large"))
#vectorstore = Chroma.from_documents(documents=chunks, embedding=AzureOpenAIEmbeddings(model="text-embedding-ada-002"))

## Test Similarity Search

In [ ]:
vectorstore.similarity_search("Amanita phalloides")

In [ ]:
# Note that providers implement different scores; Chroma here
# returns a distance metric that should vary inversely with similarity.
vectorstore.similarity_search_with_score("Amanita phalloides")

## Prepare prompt (Augmentation Step)

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    
    Question: {question} 
    
    Context: {context} 
    
    Answer:
"""
prompt = PromptTemplate.from_template(template)

## Setup retrieval

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [ ]:
	
retrieved_docs = retriever.invoke("Amanita phalloides")

len(retrieved_docs)

In [ ]:
retrieved_docs[0].page_content

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
format_docs(retriever.invoke("Amanita phalloides"))

## Build RAG chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("tell me about Amanita phalloides")

## Context Aware Text Splitter

In [ ]:
documents

In [ ]:
def llm_similarity(text1, text2):
    template = """
    Analyze the contextual relationship between the following two texts:
    
    Text 1: {text1}
    Text 2: {text2}
    
    Evaluate whether Text 2 completes or extends the context of Text 1, or if they are separate and unrelated. Assign a float score from 0 to 1, where:
    
    0 = The texts are entirely unrelated and should be split
    1 = The texts are strongly connected and belong to the same context
    
    Consider factors such as:
    
    Thematic continuity
    Logical flow
    Shared subject matter
    Narrative or argumentative progression
    Linguistic cohesion
    Provide only a single float value between 0 and 1 as your response, with up to two decimal places. For example: 0.75
    
    Ensure your answer contains nothing but the float value. Double-check your response before submitting"""
    
    prompt = PromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()

    similarity = chain.invoke({"text1": text1, "text2": text2})
    return float(similarity.replace('.\n\n', ''))

In [ ]:
llm_similarity("""The Amanita phalloides has a large and imposing epigeous (above ground) fruiting body (basidiocrap).""",
               """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all white.""")

In [ ]:
def merge_metadata(metadatas):
    merged_metadata = {}
    for metadata in metadatas:
        for key, value in metadata.items():
            if key in merged_metadata:
                merged_metadata[key] += " " + value  
            else:
                merged_metadata[key] = value
    return merged_metadata

In [ ]:
from langchain_core.documents import Document

def context_text_splitter_with_llm(documents, step_size, chunk_size, max_chunk_size):
    # Split the text
    # Ensure you have RecursiveCharacterTextSplitter defined and available
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=step_size, chunk_overlap=0)

    docs = text_splitter.split_documents(documents)

    step_chunks = [doc.page_content for doc in docs]
    step_metadata = [doc.metadata for doc in docs]

    merged_chunks = []
    merged_metadata_chunks = []

    while len(step_chunks) > 0:
        if len(''.join(step_chunks)) < chunk_size:
            break
        
        current_chunk = ''.join(step_chunks[:chunk_size//step_size])
        current_metadata = merge_metadata(step_metadata[:chunk_size//step_size])
        
        step_chunks = step_chunks[chunk_size//step_size:]
        step_metadata = step_metadata[chunk_size//step_size:]

        chunk_appended = False

        while len(step_chunks) > 0:
            next_step_chunk = step_chunks.pop(0)
            next_step_chunk_metadata = step_metadata.pop(0)

            similarity_score = llm_similarity(current_chunk, next_step_chunk)

            if similarity_score > 0.49 and len(current_chunk) + len(next_step_chunk) <= max_chunk_size:
                current_chunk += " " + next_step_chunk
                current_metadata = merge_metadata([current_metadata, next_step_chunk_metadata])
            else:
                merged_chunks.append(" " + current_chunk)
                merged_metadata_chunks.append(current_metadata)
                
                chunk_appended = True
                
                step_chunks.insert(0, next_step_chunk)
                step_metadata.insert(0, next_step_chunk_metadata)
                
                break

        if not chunk_appended:
            merged_chunks.append(" " + current_chunk)
            merged_metadata_chunks.append(current_metadata)

    if len(step_chunks) > 0:
        merged_chunks.append(' '.join(step_chunks))
        merged_metadata_chunks.append(merge_metadata(step_metadata))

    merged_docs = []
    for chunk, metadata in zip(merged_chunks, merged_metadata_chunks):
        merged_docs.append(Document(page_content=chunk, metadata=metadata))

    return merged_docs

In [ ]:
chunks = context_text_splitter_with_llm(documents, 100, 200, 1200)

In [ ]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

## Index into Chroma DB

In [ ]:
import chromadb

# Initialize ChromaDB client
client = chromadb.Client()

# Access a specific collection
collection_name = "langchain"
client.delete_collection(collection_name)

In [ ]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=AzureOpenAIEmbeddings(model="text-embedding-3-large"))

## Setup Retrieval

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("tell me about Amanita phalloides")

## Bonus Vector Similarity

In [ ]:
embeddings = AzureOpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
v1 = embeddings.embed_documents(texts=["Apple"])[0]
v2 = embeddings.embed_documents(texts=["Orange"])[0]

In [ ]:
from scipy.spatial.distance import cosine
def cosine_similarity(vec1, vec2):
    """Compute the cosine similarity between two vectors using SciPy."""
    return 1 - cosine(vec1, vec2)  # cosine function from SciPy computes the distance, not similarity

In [ ]:
cosine_similarity(v1, v2)

In [ ]:
from langchain_core.documents import Document
def context_text_splitter(documents, step_size, chunk_size, max_chunk_size):
    # Split the text
    # Ensure you have RecursiveCharacterTextSplitter defined and available
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=step_size, chunk_overlap=0)

    docs = text_splitter.split_documents(documents)

    step_chunks = [doc.page_content for doc in docs]
    step_metadata = [doc.metadata for doc in docs]

    merged_chunks = []
    merged_metadata_chunks = []

    while len(step_chunks) > 0:
        if len(''.join(step_chunks)) < chunk_size:
            break
        
        current_chunk = ''.join(step_chunks[:chunk_size//step_size])
        current_metadata = merge_metadata(step_metadata[:chunk_size//step_size])
        
        step_chunks = step_chunks[chunk_size//step_size:]
        step_metadata = step_metadata[chunk_size//step_size:]

        chunk_appended = False

        while len(step_chunks) > 0:
            next_step_chunk = step_chunks.pop(0)
            next_step_chunk_metadata = step_metadata.pop(0)

            similarity_score = cosine_similarity(embeddings.embed_query(current_chunk), embeddings.embed_query(next_step_chunk))

            if similarity_score > 0.79 and len(current_chunk) + len(next_step_chunk) <= max_chunk_size:
                current_chunk += " " + next_step_chunk
                current_metadata = merge_metadata([current_metadata, next_step_chunk_metadata])
            else:
                merged_chunks.append(" " + current_chunk)
                merged_metadata_chunks.append(current_metadata)
                
                chunk_appended = True
                
                step_chunks.insert(0, next_step_chunk)
                step_metadata.insert(0, next_step_chunk_metadata)
                
                break

        if not chunk_appended:
            merged_chunks.append(" " + current_chunk)
            merged_metadata_chunks.append(current_metadata)

    if len(step_chunks) > 0:
        merged_chunks.append(' '.join(step_chunks))
        merged_metadata_chunks.append(merge_metadata(step_metadata))

    merged_docs = []
    for chunk, metadata in zip(merged_chunks, merged_metadata_chunks):
        merged_docs.append(Document(page_content=chunk, metadata=metadata))

    return merged_docs

In [ ]:
chunks = context_text_splitter(documents, 100, 200, 1200)

In [ ]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")